# 02 · Make It Reliable

**Module 07 · Production & Observability**

On your laptop, calls succeed. In production, things go wrong in *ordinary* ways, every day:

| What goes wrong | Example |
|---|---|
| The API says "slow down" | A `429` rate limit error from OpenAI |
| The network or the server hiccups | A timeout, a `500` error |
| The same request arrives twice | A customer double-clicks, an app retries after a timeout |
| One request costs far too much | An agent in a loop, a huge input |
| The AI provider is down | Everything fails at once |

A reliable service does not avoid these. It **expects** them. Each section below handles one.

In [1]:
import json
import logging
import openai
from fastapi.testclient import TestClient

import observe
import shop
import reliability
import desk_service

client = TestClient(desk_service.app)
CLIENT = {"X-API-Key": "dev-client-key"}

shop.reset_shop()  # start with no refunds
print("Ready")

Ready


## 1. Timeouts and automatic retries (already switched on)
The OpenAI library can do two things for us. In `shop.py` we created the client with both:
- **`timeout`**: stop waiting for a call that takes too long. Without it, one stuck call can freeze a request forever.
- **`max_retries`**: if a call fails for a *temporary* reason (a rate limit, a server error, a network drop), try again. The library waits a little longer before each retry.

In [2]:
print("Timeout (seconds):", shop.client.timeout)
print("Automatic retries:", shop.client.max_retries)

Timeout (seconds): 30.0
Automatic retries: 2


## 2. Retry with backoff, by hand
Retrying is simple, but there is a trap: if a server is struggling and a thousand clients retry *immediately*, they make it worse. So we wait, and we wait **longer each time** ("exponential backoff"). A little randomness ("jitter") stops everyone from retrying at the same moment.

The library does this for OpenAI calls. Here is the idea in a few lines, so you can use it for **anything else** that can fail (your own database, another service).

*From `reliability.py`:*

```python
def retry_with_backoff(function, attempts=4, first_delay=1.0, retry_on=RETRYABLE_ERRORS):
    """Call function(). If it fails for a temporary reason, wait and try again, waiting twice as long each time.

    Waiting longer each time ("exponential backoff") gives a struggling server room to recover.
    The little random extra ("jitter") stops thousands of clients from retrying at exactly the same moment."""
    delay = first_delay

    for attempt in range(1, attempts + 1):
        try:
            return function()
        except retry_on as error:
            if attempt == attempts:
                raise  # no attempts left: let the caller decide what to do

            wait = delay + random.uniform(0, delay * 0.2)
            observe.log(logging.WARNING, "retrying", attempt=attempt, wait_seconds=round(wait, 2),
                        error=type(error).__name__)
            time.sleep(wait)
            delay = delay * 2
```

A function that fails twice and then works. The log lines show each retry, with its wait:

In [3]:
calls = {"count": 0}


def flaky():
    calls["count"] = calls["count"] + 1
    if calls["count"] < 3:
        raise ConnectionError("simulated network problem")
    return "worked on attempt " + str(calls["count"])


print(reliability.retry_with_backoff(flaky, first_delay=0.3, retry_on=(ConnectionError,)))

{"time": "2026-11-22T19:15:51+00:00", "level": "WARNING", "event": "retrying", "attempt": 1, "wait_seconds": 0.34, "error": "ConnectionError"}


{"time": "2026-11-22T19:15:52+00:00", "level": "WARNING", "event": "retrying", "attempt": 2, "wait_seconds": 0.68, "error": "ConnectionError"}


worked on attempt 3


And one that never works. After the last attempt, the error is **passed on**, because retrying forever is not a plan:

In [4]:
def always_fails():
    raise ConnectionError("simulated outage")


try:
    reliability.retry_with_backoff(always_fails, attempts=3, first_delay=0.2, retry_on=(ConnectionError,))
except ConnectionError as error:
    print("Gave up after 3 attempts:", error)

{"time": "2026-11-22T19:15:52+00:00", "level": "WARNING", "event": "retrying", "attempt": 1, "wait_seconds": 0.24, "error": "ConnectionError"}


{"time": "2026-11-22T19:15:53+00:00", "level": "WARNING", "event": "retrying", "attempt": 2, "wait_seconds": 0.42, "error": "ConnectionError"}


Gave up after 3 attempts: simulated outage


**Retry only temporary errors.** A `400 Bad Request` (your request is wrong) will fail the same way every time. The `RETRYABLE_ERRORS` list in `reliability.py` contains only the temporary kinds.

## 3. A fallback: never leave the customer with an error
Retries are not enough when the provider is *down*. The question becomes: what does the customer see?

In `process_email` (notebook 1), anything that goes wrong is caught, logged as an `ERROR`, and the email is **handed to a person**. Let us break the desk on purpose and see:

In [5]:
real_handle_email = desk_service.desk.handle_email


def broken_desk(email, guarded=True):
    raise ConnectionError("simulated OpenAI outage")


desk_service.desk.handle_email = broken_desk   # break the desk on purpose

response = client.post("/emails", json={"email": "Where is my order A1002?"}, headers=CLIENT)
print("HTTP", response.status_code)
print(json.dumps(response.json(), indent=2))

desk_service.desk.handle_email = real_handle_email   # put it back

{"time": "2026-11-22T19:15:53+00:00", "level": "ERROR", "event": "desk_failed", "request_id": "2e39b79f9113", "error": "ConnectionError: simulated OpenAI outage"}


{"time": "2026-11-22T19:15:53+00:00", "level": "INFO", "event": "request_finished", "request_id": "2e39b79f9113", "status": "error", "duration_ms": 3, "model_calls": 0, "cost_usd": 0.0}


HTTP 200
{
  "request_id": "2e39b79f9113",
  "status": "human",
  "reply": "",
  "route": "",
  "note": "The desk had a problem, so a person will answer this email.",
  "refunds_issued": [],
  "sent_for_approval": [],
  "cost_usd": 0.0,
  "latency_ms": 3
}


The customer's app got a normal answer (`status: human`), not a crash. The `ERROR` line in the log is what wakes up the person on call, and the trace is marked as an error.

**Degrade gracefully.** Decide in advance what the service does when the AI is unavailable. For a support desk, "a person answers" is a good fallback.

## 4. The same request twice: idempotency
A request that **changes something** (a refund!) must be safe to repeat. Customers double-click. Apps retry after a timeout, because they do not know whether the first request worked.

**Idempotency** means: doing it twice has the same effect as doing it once. The caller attaches a unique `Idempotency-Key` to the request. If we have seen the key, we return the **saved answer** and do nothing again.

*From `reliability.py`:*

```python
class IdempotencyStore:
    """Remember the answer to a request, so a repeated request gets the SAME answer and does nothing twice.

    A customer double-clicks, a phone loses signal and resends, a client retries after a timeout: the same
    request can arrive more than once. The caller attaches a unique key to each request. If we have seen the key,
    we return the saved answer instead of, for example, refunding the order a second time.

    (In a real system this lives in a database such as Redis and expires after a day. Here it is a dictionary.)"""

    def __init__(self):
        self.saved = {}
        self.lock = threading.Lock()

    def get(self, key):
        with self.lock:
            return self.saved.get(key)

    def put(self, key, answer):
        with self.lock:
            self.saved[key] = answer
```

The service uses it in the `/emails` endpoint. We send the *same* refund request twice with the *same* key:

In [6]:
refund_email = "The left earbud on my headphones (order A1001) stopped working. I would like my money back."
headers = {"X-API-Key": "dev-client-key", "Idempotency-Key": "refund-demo-1"}

first = client.post("/emails", json={"email": refund_email}, headers=headers).json()
second = client.post("/emails", json={"email": refund_email}, headers=headers).json()

print("First  request id:", first["request_id"], " refunds issued:", first["refunds_issued"])
print("Second request id:", second["request_id"], " refunds issued:", second["refunds_issued"])
print("Same answer, and the desk ran only once:", first["request_id"] == second["request_id"])
print("Refunds in the shop's data:", shop.REFUNDS)

{"time": "2026-11-22T19:15:57+00:00", "level": "INFO", "event": "audit:refund_issued", "request_id": "06f1378e2a86", "order_id": "A1001", "amount": 79.0, "approved_by": "agent (under the automatic limit)"}


{"time": "2026-11-22T19:15:59+00:00", "level": "INFO", "event": "request_finished", "request_id": "06f1378e2a86", "status": "ok", "duration_ms": 5405, "model_calls": 7, "cost_usd": 0.001599}


{"time": "2026-11-22T19:15:59+00:00", "level": "INFO", "event": "reply_graded", "request_id": "3764677488ec", "judged_request": "06f1378e2a86", "score": 5}


{"time": "2026-11-22T19:15:59+00:00", "level": "INFO", "event": "request_finished", "request_id": "3764677488ec", "status": "ok", "duration_ms": 957, "model_calls": 1, "cost_usd": 0.00148}


First  request id: 06f1378e2a86  refunds issued: ['A1001']
Second request id: 06f1378e2a86  refunds issued: ['A1001']
Same answer, and the desk ran only once: True
Refunds in the shop's data: [{'order_id': 'A1001', 'amount': 79.0}]


One refund, and the second call was answered from memory in a fraction of a second.

**Defence in depth.** What if the retry has *no* key? Then the desk runs again, but the **refund tool itself refuses a second refund** for the same order (we added that guard in Module 04). Two layers protect the money:

In [7]:
third = client.post("/emails", json={"email": refund_email}, headers=CLIENT).json()   # no Idempotency-Key this time

print("Reply:", third["reply"])
print("Refunds in the shop's data (still one):", shop.REFUNDS)

{"time": "2026-11-22T19:16:04+00:00", "level": "INFO", "event": "request_finished", "request_id": "7e135701a793", "status": "ok", "duration_ms": 4781, "model_calls": 7, "cost_usd": 0.001591}


Reply: A refund for your order A1001 has been issued. Thank you for your understanding!
Refunds in the shop's data (still one): [{'order_id': 'A1001', 'amount': 79.0}]


## 5. A cost cap on every request
A request that spends too much is a bug or an attack. We cannot predict *which* request, so every request gets a **budget**. `observe.py` adds up the cost of each model call as it happens, and before the *next* call it checks the total. Over the limit? The request stops, and the safe fallback hands the email to a person.

The normal limit is $0.02, about ten times what a typical email costs. To see it work, we set it absurdly low for one request:

In [8]:
print("Normal limit: $" + str(observe.MAX_COST_PER_REQUEST))

observe.MAX_COST_PER_REQUEST = 0.0001   # a tiny budget, just for this demonstration

response = client.post("/emails", json={"email": "Where is my order A1002?"}, headers=CLIENT).json()
print("Status:", response["status"])
print("Note:", response["note"])
print("Cost of the request before it was stopped: $" + str(response["cost_usd"]))

observe.MAX_COST_PER_REQUEST = 0.02   # back to normal

Normal limit: $0.02


{"time": "2026-11-22T19:16:12+00:00", "level": "ERROR", "event": "desk_failed", "request_id": "838e473162cf", "error": "BudgetExceeded: This request has spent $0.0001, which is the limit of $0.0001."}


{"time": "2026-11-22T19:16:12+00:00", "level": "INFO", "event": "request_finished", "request_id": "838e473162cf", "status": "error", "duration_ms": 7224, "model_calls": 4, "cost_usd": 0.000122}


Status: human
Note: The desk had a problem, so a person will answer this email.
Cost of the request before it was stopped: $0.000122


The log shows *why* it stopped (`BudgetExceeded`, with the amounts), so nobody has to guess.

## Rules of thumb
- **Set a timeout on everything that waits.** No timeout means one slow call can block a whole request.
- **Retry only temporary errors, with backoff and a limit.**
- **Make every action that changes something idempotent**, especially money.
- **Give every request a budget** (steps, tokens and dollars).
- **Decide the fallback in advance.** "A person answers" beats "500 Internal Server Error".
- **Layer your protections.** The idempotency key *and* the refund tool's own check.

Next: **what exactly happened inside one request?** That is what logging and tracing are for.